# AXI ARC-AGI-3 probe (free: no model, no API key, no money)
**What this is.** ARC-AGI-3 is a set of small interactive games whose rules are not told to the player. Before I build Test 23 (Gemini alone against Gemini helped by AXI on these games) I need to see, on a machine that can reach PyPI, what the toolkit really looks like: the list of games, what one observation contains, which actions exist, and how fast a step is. **This notebook calls no AI model and costs nothing.** It only installs the public `arc-agi` toolkit, plays a few random moves in a few games, and prints what it saw.

Use the free **CPU** runtime. Run each cell one at a time, top to bottom. At the end, copy the whole output of Cells 3 to 6 into the chat (Cell 7 saves the same to a file you can download).

### Cell 1: install the toolkit (about 1 minute)

In [ ]:
!pip -q install arc-agi arcengine 2>&1 | tail -3

### Cell 2: what is installed

In [ ]:
import importlib, pkgutil, sys
for name in ("arc_agi", "arcengine"):
    try:
        m = importlib.import_module(name); print(name, "ok | version", getattr(m, "__version__", "?"), "| file", getattr(m, "__file__", "?"))
        print("   names:", [n for n in dir(m) if not n.startswith("_")][:40])
    except Exception as e: print(name, "FAILED:", type(e).__name__, e)

### Cell 3: open the arcade and list the games (no key needed for this; if it asks for one, the output will say so)

In [ ]:
import os, time, json
REPORT = {}
try:
    from arc_agi import Arcade
    arc = Arcade()
    for attr in ("get_environments", "list_games", "environments", "games"):
        if hasattr(arc, attr):
            try:
                v = getattr(arc, attr); v = v() if callable(v) else v
                ids = [getattr(e, "game_id", getattr(e, "id", e)) for e in v]
                print(attr, "->", len(ids), "games:", ids[:30]); REPORT["games"] = [str(i) for i in ids]; break
            except Exception as e: print(attr, "failed:", type(e).__name__, e)
    print("Arcade methods:", [n for n in dir(arc) if not n.startswith("_")])
except Exception as e: print("ARCADE FAILED:", type(e).__name__, e)

### Cell 4: make one game and look at what the first observation holds

In [ ]:
def describe(o, depth=0, name="obs"):
    pad = "  " * depth
    if o is None: print(pad + name, "= None"); return
    t = type(o).__name__
    if isinstance(o, (int, float, str, bool)): print(pad + f"{name}: {t} = {str(o)[:80]}"); return
    if hasattr(o, "shape"): print(pad + f"{name}: {t} shape {tuple(o.shape)}"); return
    if isinstance(o, (list, tuple)): print(pad + f"{name}: {t} len {len(o)}" + (f" | first item {type(o[0]).__name__}" if o else "")); return
    if isinstance(o, dict):
        print(pad + f"{name}: dict keys {list(o)[:15]}"); return
    attrs = [a for a in dir(o) if not a.startswith("_") and not callable(getattr(o, a, None))]
    print(pad + f"{name}: {t} fields {attrs[:25]}")
    for a in attrs[:12]:
        if depth < 1:
            try: describe(getattr(o, a), depth + 1, a)
            except Exception as e: print(pad + "  ", a, "unreadable", e)
GAME = REPORT["games"][0] if REPORT.get("games") else "ls20"
try:
    env = arc.make(GAME); print("made game", GAME, "->", type(env).__name__)
    print("env methods:", [n for n in dir(env) if not n.startswith("_")])
    obs = getattr(env, "observation_space", None) or getattr(env, "reset", lambda: None)()
    describe(obs)
except Exception as e: print("MAKE FAILED:", type(e).__name__, e)

### Cell 5: which actions exist

In [ ]:
try:
    from arcengine import GameAction
    acts = [a for a in GameAction]; print("actions:", [(a.name, getattr(a, "value", None)) for a in acts])
    REPORT["actions"] = [a.name for a in acts]
except Exception as e: print("ACTIONS FAILED:", type(e).__name__, e)

### Cell 6: play 30 random moves in up to 3 games and time each step (this is the speed of the game itself, with no model)

In [ ]:
import random
random.seed(1); REPORT["plays"] = []
simple = [a for a in acts if a.name not in ("RESET",) and not getattr(a, "is_complex", lambda: False)()] if "acts" in dir() else []
for g in (REPORT.get("games") or [GAME])[:3]:
    try:
        env = arc.make(g); times = []; last = None; levels = None
        for i in range(30):
            a = random.choice(simple); t0 = time.perf_counter()
            last = env.step(a); times.append(time.perf_counter() - t0)
        state = getattr(last, "state", None); levels = getattr(last, "levels_completed", None)
        print(f"{g}: 30 steps, {sum(times)/30*1000:.1f} ms a step | last state {state} | levels completed {levels}")
        REPORT["plays"].append(dict(game=g, ms_per_step=sum(times)/30*1000, state=str(state), levels=str(levels)))
        if g == (REPORT.get("games") or [GAME])[0]: describe(last, name="step result")
    except Exception as e: print(g, "PLAY FAILED:", type(e).__name__, e)

### Cell 7: save the report (download it and send it, together with the printout above)

In [ ]:
from google.colab import files
json.dump(REPORT, open("arc_probe_report.json", "w"), indent=1); files.download("arc_probe_report.json"); print(json.dumps(REPORT)[:1500])

### Done
Send me the printout. Random moves are not expected to solve anything: this only shows how the games look from code, so that Test 23 can be written against the real thing.